# data processing
for ruiyi data structure

## imports

In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
import sys
import os
from pathlib import Path
import configparser
config = configparser.ConfigParser()
config.read_file(open('privateconfig'))
resdir = Path(config['Datafolder']['data'])
workdir = Path(config['Codefolder']['workspace'])
os.chdir(workdir)

In [3]:
# analysis
from scipy.io import loadmat
from sklearn.decomposition import FastICA
from sklearn.datasets import make_regression
from sklearn.model_selection import KFold
from sklearn.linear_model import LassoCV, Lasso
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr


In [4]:
# misc
import pickle
from collections import defaultdict

In [5]:
# task
from env_config import Config
from firefly_task import ffacc_real
# from monkey_functions import *
# from InverseFuncs import *
from stable_baselines3 import TD3
import torch


In [6]:
from neural_plot_ult import *
import time
tic=time.time()
import warnings
warnings.filterwarnings('ignore')

# Pre IRC

convert the mat data file (with neural data) into (states, actions, tasks) for IRC.


## prepare

In [7]:
# const
vgain = 200 # forward gain, 200 cm
wgain = 90 # angular gain, 90 degree
worldscale=200


## from ruiyi data file

In [8]:
# load 
file=Path(Path(resdir/'variable_data_both_test.pkl'))
df=pd.read_pickle(file)

In [9]:
# df=df.head(55) # for quick testing

In [10]:
df.density.unique()

array([0.005 , 0.0001])

In [11]:
df.monk_name.unique()

array(['m53'], dtype=object)

In [12]:
m = 'm53'
session=df.sess_id.unique()
session

array(['s105', 's113', 's115', 's124', 's91', 's93', 's97'], dtype=object)

## IRC input data (state, action, task)

In [13]:
sessdata=defaultdict(list)

for sess in session:
    states, actions, tasks=[],[],[]
    sessdf=df[df.sess_id==sess]
    trial_idces=sessdf.trial_idx

    for trial_idx in trial_idces:
        trialdf=sessdf[sessdf.trial_idx==trial_idx]
        trialdata=trialdf.iloc[0]
        # task
        taskx = -(trialdata.tar_x - trialdata.mx[0]).astype('float32'); tasky = (trialdata.tar_y - trialdata.my[0]).astype('float32')
        tasks.append([tasky/worldscale,taskx/worldscale])
        # actions
        trialaction=np.stack([trialdata.mv_ds,trialdata.mw_ds]).T
        trialaction[:,0]=trialaction[:,0]/vgain
        trialaction[:,1]=trialaction[:,1]/wgain
        actions.append(trialaction.astype('float32'))

        # states from run the actions
        px, py, heading, v, w = 0,0,0,0,0
        log=[]
        for a in trialaction:
            px, py, heading, v, w=state_step2(px, py, heading, v, w, a, dt=0.1,userad=True)
            log.append([px, py, heading, v, w])
        px, py, heading, v, w=state_step2(px, py, heading, v, w, a, dt=0.1,userad=True)
        log.append([px, py, heading, v, w])
        trialstates=np.array(log)[1:]
        # plt.plot(trialstates[:,0],trialstates[:,1])
        # plt.scatter(tasky/worldscale,taskx/worldscale)
        # plt.plot(trialdata.my, trialdata.mx);plt.show()
        # plt.plot(trialdata.body_theta_ds-pi/2)
        # plt.plot(np.cumsum(trialstates[:,4])*0.1)
        # plt.plot(trialstates[:,2]);plt.show()
        states.append(trialstates.astype('float32'))

        sessdata['sess_id'].append(sess)
        sessdata['trial_idx'].append(trial_idx)
        sessdata['state'].append(trialstates.astype('float32'))  
        sessdata['action'].append(trialaction.astype('float32'))  
        sessdata['task'].append([tasky/worldscale,taskx/worldscale])  

tmp=pd.DataFrame(sessdata)
df = pd.merge(df, tmp, on=['trial_idx','sess_id'], how='inner')

## Compute belief 

In [14]:
# model estimated likelihood (negative log likelihood)
torch.manual_seed(42)
arg = Config()

env = ffacc_real.FireFlyPaper(arg)
env.debug=True
phi = torch.tensor([[0.5],
                    [pi/2],
                    [0.001],
                    [0.001],
                    [0.001],
                    [0.001],
                    [0.13],
                    [0.001],
                    [0.001],
                    [0.001],
                    [0.001],
                    ])

agent_ = TD3.load(workdir/'trained_agent/paper')
agent = agent_.actor.mu.cpu()

In [15]:
thetas={}
for idensity in range(4):
    savename = Path(resdir/f'{m}_mat_ruiyi/schro_normal1102_den1packed')
    # savename = datapath.parent/(f'{m}_{idensity}'+datapath.name)
    invfile=savename
    # print(datapath.name)
    finaltheta, finalcov, err = process_inv(
        invfile, removegr=False, usingbest=False)
    print(finaltheta[:4])
    # finaltheta[0]=1
    # finaltheta[1]=1.3
    # finaltheta[1]=0.5
    # finaltheta[1]=0.2
    thetas[idensity]=finaltheta


/Users/yc/Documents/lab_data/m53_mat_ruiyi/schro_normal1102_den1packed
using ind:  -1 final logll :  14.32904863357544
tensor([[0.7835],
        [1.8350],
        [1.4835],
        [0.3741]])
/Users/yc/Documents/lab_data/m53_mat_ruiyi/schro_normal1102_den1packed
using ind:  -1 final logll :  14.32904863357544
tensor([[0.7835],
        [1.8350],
        [1.4835],
        [0.3741]])
/Users/yc/Documents/lab_data/m53_mat_ruiyi/schro_normal1102_den1packed
using ind:  -1 final logll :  14.32904863357544
tensor([[0.7835],
        [1.8350],
        [1.4835],
        [0.3741]])
/Users/yc/Documents/lab_data/m53_mat_ruiyi/schro_normal1102_den1packed
using ind:  -1 final logll :  14.32904863357544
tensor([[0.7835],
        [1.8350],
        [1.4835],
        [0.3741]])


In [16]:
denslookup={0.0001:0, 0.0005:1, 0.001:2,  0.005:3}
thisdensity=trialdf.density.item()
denslookup[thisdensity],thisdensity

(0, 0.0001)

In [17]:
# df belief

def lltrial(state, action, task, finaltheta, samples=5):
    with torch.no_grad():
        return monkeyloss_(agent, action, np.array(task).reshape(1,-1), phi, finaltheta, env, action_var=0.01, num_iteration=1, states=state, samples=samples, gpu=False).item()

belief_df=defaultdict(list)

for sess in session:
    sessdf=df[df.sess_id==sess]
    state, action, task = df.state.to_list(), df.action.to_list(),df.task.to_list()
    trial_idces=sessdf.trial_idx
    for trial_idx in trial_idces:
        trialdf=sessdf[sessdf.trial_idx==trial_idx]
        trialdata=trialdf.iloc[0]
        state, action, task = trialdata.state, trialdata.action, trialdata.task
        thisdensity=trialdf.density.item()
        theta=thetas[denslookup[thisdensity]]

        _, _, ep_belief, ep_rawcov = run_trials(agent=agent, 
                                                env=env, phi=phi, theta=theta,          task=task, ntrials=1,
                                                pert=None, given_obs=None, return_belief=True, given_action=action, given_state=state)
        # trial info
        belief_df['sess_id'].append(sess)
        belief_df['trial_idx'].append(trial_idx)

        # belief
        if len(state)<5: # 
            belief_df['belief'].append(np.nan)
            belief_df['rawcov'].append(np.nan)
        else:
            init=torch.tensor(state[0]).reshape(-1,1)
            trial_belief=(ep_belief[0]-ep_belief[0][0]+init)
            belief_df['belief'].append(np.array(trial_belief)[:,:,0])
            belief_df['rawcov'].append(np.array(ep_rawcov[0]))
    

notify('compute belief complete')


tmp=pd.DataFrame(belief_df)
tmp=tmp.rename(columns={'cov': 'rawcov'})
df = pd.merge(df, tmp, on=['trial_idx','sess_id'], how='inner')

# check

In [18]:
# i=3
# plt.plot(df.belief[i][:,0],df.belief[i][:,1])
# plt.plot(df.state[i][:,0],df.belief[i][:,1])


In [19]:
# unpack the belief state
df['bmx']=df.apply(lambda x:x.belief[:,1]*worldscale*-1, axis=1)
df['bmy']=df.apply(lambda x:x.belief[:,0]*worldscale, axis=1)
df['heading']=df.apply(lambda x: x.state[:,2]*180/pi, axis=1)
df['belief_heading']=df.apply(lambda x: x.belief[:,2]*180/pi, axis=1)
# df['timer']=df.apply(lambda x: np.arange(len(x.mx)), axis=1)
# df['countdown']=df.apply(lambda x: np.flip(-np.arange(len(x.mx)), axis=0), axis=1)

In [20]:
# plt.plot(row.body_theta_ds*180/pi)
# plt.plot(np.cumsum(row.mw_ds)*0.1+90)
# plt.plot(row.heading+90)
# plt.show()

In [21]:
# row=df.iloc[i]
# i+=1
# plt.plot(row.body_theta_ds)
# plt.plot(np.cumsum(row.mw_ds)+90)
# plt.show()

# plt.plot(row.heading+90)
# plt.plot(row.belief_heading+90)

In [22]:
sessdata=defaultdict(list)

for sess in session:
    sessdf=df[df.sess_id==sess]
    states, actions, tasks = sessdf.state.to_list(), sessdf.action.to_list(),sessdf.task.to_list()
    beliefs,rawcovs=sessdf['belief'].to_list(),sessdf['rawcov'].to_list()

    sess_latentff_hori, sess_latentff_vert = [], []
    for ep_beliefs, ep_rawcovs, task in zip(beliefs, rawcovs, tasks): # process for each trial
        mx, my, body_theta,  mv, mw = zip(*ep_beliefs)
        body_theta = -(np.cumsum(mw) * 0.1-pi/2)
        body_x, body_y = np.asarray(my).reshape(-1).astype('float') * \
            worldscale, np.asarray(mx).reshape(-1).astype('float')*worldscale

        fx, fy = task[1]*worldscale, task[0]*worldscale
        rel_dist = ((fx-body_x)**2+(fy-body_y)**2)**0.5
        hor_theta_, ver_theta_ = convert_location_to_angle(abs(np.array(rel_dist)).reshape(-1).astype('float'), np.array(fx).reshape(-1).astype('float'), np.array(fy).reshape(-1).astype('float'),
                                                           body_theta.astype('float'), body_x.astype(
                                                               'float'), body_y.astype('float'),
                                                           np.array(rel_dist).reshape(-1).astype('float'), # use the true eye positions to remove pre saccade movement and after overshooting eye movement
                                                           np.array(rel_dist).reshape(-1).astype('float'), DT=0.1, remove_pre=False, remove_post=False)
        # plt.plot(hor_theta_, ver_theta_, 'g')
        sess_latentff_hori.append(hor_theta_)
        sess_latentff_vert.append(ver_theta_)
    sessdata['belief_ff_hori']+=[a.reshape(-1) for a in sess_latentff_hori]
    sessdata['belief_ff_vert']+=[a.reshape(-1) for a in sess_latentff_vert]
    

# TODO temp need change to merge
df['belief_ff_hori']=sessdata['belief_ff_hori']
df['belief_ff_vert']=sessdata['belief_ff_vert']

In [ ]:
df.columns

df.to_pickle('m53__ruiyi_test.pkl')
!open .

In [34]:
df

,monk_name,sess_id,trial_idx,tar_x,tar_y,tar_r,tar_ang,error,error_sign,density,...,action,task,belief,rawcov,bmx,bmy,heading,belief_heading,belief_ff_hori,belief_ff_vert
0,m53,s105,0,94.637077,132.955643,163.197357,-35.443050,71.553688,71.553688,0.0050,...,"[[0.0009412017, -2.134155e-05], [0.1341776, 0....","[0.6768347930908203, -0.47293216705322266]","[[9.4120165e-05, -1.0042811e-10, -2.134155e-06...","[[[9.0249996e-05, 0.0, 0.0, 0.0, 0.0], [0.0, 0...","[2.0085622e-08, 2.0085622e-08, -0.025558451, -...","[0.018824033, 0.018824033, 5.01542, 31.191658,...","[-0.00012227807, 0.09816964, 0.52573967, 0.557...","[-0.00012227807, -0.00012227807, 0.5868631, 1....","[-34.9724150169928, -35.65768789642017, -37.51...","[-3.4656453698613428, -3.4656453698613428, -3...."
1,m53,s105,1,27.833385,260.277100,261.761078,-6.103874,77.000404,-77.000404,0.0001,...,"[[0.85566366, 9.44973e-06], [0.86103404, 4.214...","[1.3180409240722657, -0.13919713973999023]","[[0.085566364, 4.042522e-08, 9.4497295e-07, 0....","[[[9.0249996e-05, 0.0, 0.0, 0.0, 0.0], [0.0, 0...","[-8.085044e-06, -8.085044e-06, -8.085044e-06, ...","[17.113274, 17.113274, 17.113274, 43.905743, 7...","[5.4142958e-05, 0.00029559116, -0.052954607, -...","[5.4142958e-05, 5.4142958e-05, 0.35516682, 0.8...","[-6.438734517961744, -6.794088769909997, -7.25...","[-2.3084859176684076, -2.3084859176684076, -2...."
2,m53,s105,2,45.550262,269.933105,273.749359,-9.578232,22.679350,-22.679350,0.0050,...,"[[0.6793649, -0.106349036], [0.8538944, -0.232...","[1.3711819458007812, -0.22804691314697265]","[[0.06793521, -0.00036124562, -0.010634904, 0....","[[[9.0249996e-05, 0.0, 0.0, 0.0, 0.0], [0.0, 0...","[0.07224912, 0.07224912, 0.17696255, 0.8059935...","[13.587043, 13.587043, 27.498964, 49.74178, 76...","[-0.60933506, -1.9434925, -3.349793, -4.645347...","[-0.60933506, -0.60933506, -1.4719062, -2.9868...","[-8.56862969649535, -6.3718974254466785, -4.04...","[-2.164365676672617, -2.164365676672617, -2.28..."
3,m53,s105,3,181.748077,314.017853,362.821686,-30.061516,155.751068,-155.751068,0.0001,...,"[[0.58245146, -0.00553408], [0.73144436, -0.07...","[1.5854261779785157, -0.9087875366210938]","[[0.058245145, -1.6116664e-05, -0.000553408, 0...","[[[9.0249996e-05, 0.0, 0.0, 0.0, 0.0], [0.0, 0...","[0.0032233328, 0.0032233328, -0.11789331, -0.1...","[11.649029, 11.649029, 31.904932, 35.51575, 49...","[-0.031707942, -0.46094608, -1.1831813, -3.115...","[-0.031707942, -0.031707942, 0.6538077, 1.3169...","[-30.31265570121677, -30.568931229064816, -32....","[-1.6116171549616667, -1.6116171549616667, -1...."
4,m53,s105,4,-70.343796,203.802917,215.601196,19.042412,37.200645,37.200645,0.0001,...,"[[0.75329, -0.34928036], [0.7533197, -0.352414...","[1.0310530853271485, 0.35132823944091796]","[[0.07531369, -0.0013154133, -0.03492804, 0.75...","[[[9.0249996e-05, 0.0, 0.0, 0.0, 0.0], [0.0, 0...","[0.26308265, 0.26308265, 0.26308265, 0.9377413...","[15.062737, 15.062737, 15.062737, 20.843494, 3...","[-2.001229, -4.020417, -6.0389123, -7.4881864,...","[-2.001229, -2.001229, -6.927744, -10.388212, ...","[22.24656005038865, 29.19226137964466, 34.6719...","[-2.809884120015803, -2.809884120015803, -2.80..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6714,m53,s97,614,-133.396240,312.756226,340.016174,23.099129,247.639893,-247.639893,0.0001,...,"[[0.45799407, -0.13248685], [0.8426683, -0.380...","[1.5816641235351563, 0.6678456878662109]","[[0.045798067, -0.0003033865, -0.013248685, 0....","[[[9.0249996e-05, 0.0, 0.0, 0.0, 0.0], [0.0, 0...","[0.0606773, 0.0606773, 0.61326003, 2.6427188, ...","[9.159614, 9.159614, 25.647112, 46.282898, 58....","[-0.7590937, -2.9373267, -5.870905, -8.825025,...","[-0.7590937, -0.7590937, -4.5980897, -8.153767...","[25.67766536254856, 31.694891266600546, 38.694...","[-1.7099106711029213, -1.7099106711029213, -1...."
6715,m53,s97,615,13.117939,141.383331,141.990585,-5.300888,126.454773,126.454773,0.0050,...,"[[0.92726296, -0.6127059], [0.92609924, 

In [24]:
# i=6
# plot_gradient_line(plt.gca(), -df.iloc[i].bmx, df.iloc[i].bmy, cmap_state)
# plot_gradient_line(plt.gca(), df.iloc[i].mx, df.iloc[i].my, cmap_eye)

In [25]:
# # angle from start
# def get_angle_from_start(row):
#     return np.arctan2((np.array(row.my)), (np.array(row.mx)))

# df['angle_from_start']=df.apply(get_angle_from_start, axis=1)

# def get_belief_angle_from_start(row):
#     return np.arctan2((np.array(row.bmy)), (np.array(row.bmx)))

# df['belief_angle_from_start']=df.apply(get_angle_from_start, axis=1)


In [26]:
# # check len
# random_rows = df.sample(n=3)

# # Iterate over the columns
# for col in random_rows.columns:
#     try:
#         # Get the column values for the selected rows
#         col_values = random_rows[col]
#         # Calculate the length of each column value
#         lengths = col_values.apply(lambda x: len(x))
#         # Print column name and the length of each column value
#         print(f"Column: {col}, Lengths: {lengths.tolist()}")
#     except: continue


In [27]:
# # check size
# random_rows = df.sample(n=3)

# # Iterate over the columns
# for col in random_rows.columns:
#     try:
#         # Get the column values for the selected rows
#         col_values = random_rows[col]
#         # Calculate the length of each column value
#         lengths = col_values.apply(lambda x: (x.shape))
#         # Print column name and the length of each column value
#         print(f"Column: {col}, Lengths: {lengths.tolist()}")
#     except: continue

## the varialbes we have:

In [28]:
# # state, change coord example
# trialdf=df.iloc[10]
# mx, my,mw, fx,fy=trialdf.mx, trialdf.my, trialdf.mw, trialdf.fx, trialdf.fy
# mx, my,mw, fx,fy=[np.array(a) for a in [mx, my,mw, fx,fy]]
# sx = np.ones_like(fx)
# sy = np.ones_like(fy)
# if my.size > 0:
#     fx = np.ones_like(fx) * fx[0]
#     fy = np.ones_like(fy) * fy[0]
#     sx *= mx[-1]
#     sy *= my[-1]
#     my = my + 30
#     fy = fy + 30
#     sy = sy + 30
# dx = fx - mx; dy = fy - my
# rel_dist = np.sqrt(dx**2 + dy**2); rel_ang = np.rad2deg(np.arctan2(dy, dx))
# # rel_dist_stop = np.sqrt((sx - mx)**2 + (sy - my)**2)
# # abs_dist = np.sqrt(mx**2 + my**2); abs_ang = np.rad2deg(np.arctan2(my, mx))
# heading = np.deg2rad(np.cumsum(mw*-1) * 0.1 + 90) 

# latent_ff_hori, latent_ff_vert = convert_location_to_angle(
#     rel_dist,
#     fx,
#     fy,
#     heading,
#     mx,
#     my,
#     None, # not needed if remove pre = false
#     None,
#     remove_pre=False
# )
# # plt.scatter(latent_ff_hori,trialdf.ff_hori); plt.title('state');plt.axis('equal'); plt.show()


In [29]:
# # # belief chnage coord example
# trialdf=df.iloc[23]

# bmx, bmy, bmw, fx,fy = trialdf.bmx, trialdf.bmy, np.array(trialdf.mw), trialdf.fx, trialdf.fy
# ep_beliefs=trialdf.belief

# mx, my, body_theta,  mv, mw = zip(*ep_beliefs)
# mx, my,mw, fx,fy=[np.array(a) for a in [mx, my,mw, fx,fy]]
# mx,my=my*worldscale, mx*worldscale

# # plt.scatter(mx, bmx); plt.title('beliefmx vs bmx');plt.axis('equal'); plt.show()

# sx = np.ones_like(fx)
# sy = np.ones_like(fy)
# if my.size > 0:
#     fx = np.ones_like(fx) * fx[0]
#     fy = np.ones_like(fy) * fy[0]
#     sx *= mx[-1]
#     sy *= my[-1]
#     my = my + 30
#     fy = fy + 30
#     sy = sy + 30
# dx = fx - mx; dy = fy - my
# rel_dist = np.sqrt(dx**2 + dy**2); rel_ang = np.rad2deg(np.arctan2(dy, dx))
# # rel_dist_stop = np.sqrt((sx - mx)**2 + (sy - my)**2)
# # abs_dist = np.sqrt(mx**2 + my**2); abs_ang = np.rad2deg(np.arctan2(my, mx))

# heading2=-(np.cumsum(mw) * 0.1-pi/2)

# latent_ff_hori, latent_ff_vert = convert_location_to_angle(
#     rel_dist,
#     fx,
#     fy,
#     heading2,
#     mx,
#     my,
#     None, # not needed if remove pre = false
#     None,
#     remove_pre=False
# )
# # plt.scatter(latent_ff_hori,trialdf.belief_ff_hori); plt.title('belief');plt.axis('equal'); plt.show()


In [30]:
# # convert belief rawcov example
# trialdf=df.iloc[23]

# bmx, bmy, bmw, fx,fy = trialdf.bmx, trialdf.bmy, np.array(trialdf.mw), trialdf.fx, trialdf.fy
# rawcov=trialdf['rawcov'][:,:2,:2]


# belief_heading=trialdf.belief_heading
# rotdegree=belief_heading+180
# relativeposrawcov=[]
# for degree, thisrawcov in zip(rotdegree, rawcov):
#     R=np.array([[np.cos(-degree/180*pi),-np.sin(-degree/180*pi)],[np.sin(-degree/180*pi),np.cos(-degree/180*pi)]])
#     relativeposrawcov.append(R.T@thisrawcov[:2,:2]@R)
# relativeposrawcov=np.stack(relativeposrawcov)
# relativeposrawcov.shape
# rotdegree

In [31]:
# # rotate belief cov df
# def fun(trialdf):
#     if trialdf.fullon==1: # always on target, assign zero uncertainty
#         return np.ones_like(trialdf['rawcov'][:,:2,:2])*1e-6
#     cov=trialdf['rawcov'][:,:2,:2]
#     belief_heading=trialdf.belief_heading
#     rotdegree=belief_heading+180
#     relativeposcov=[]
#     for degree, thiscov in zip(rotdegree, cov):
#         R=np.array([[np.cos(-degree/180*pi),-np.sin(-degree/180*pi)],[np.sin(-degree/180*pi),np.cos(-degree/180*pi)]])
#         relativeposcov.append(R.T@thiscov[:2,:2]@R)
#     relativeposcov=np.stack(relativeposcov)*worldscale*worldscale
#     return relativeposcov
# df['relcov']=df.apply(fun, axis=1)

In [32]:
# date='0918'
# df.to_pickle(resdir/f'{date}_m51df.pkl')
# notify(f'saved: {date}_m51df.pkl')
# print(f'saved: {date}_m51df.pkl')

In [33]:
# # with neuron variance as PPC_var
# date='1014'
# df.to_pickle(resdir/f'{date}_m51df.pkl')
# notify(f'saved: {date}_m51df.pkl')
# print(f'saved: {date}_m51df.pkl')